Imports

In [ ]:
# Core
import os
import re
import random
import warnings
from pathlib import Path
from collections import defaultdict
import sys

# Data handling
import numpy as np
import pandas as pd

# Image processing 
from PIL import Image
import cv2

# Visualization
import matplotlib.pyplot as plt

# Deep Learning / Embeddings
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torchvision.transforms as transforms
from transformers import (
    AutoModel,              # DINOv2 for B2 embeddings
    AutoImageProcessor,     # DINOv2 processor
    TrOCRProcessor,         # B3 OCR
    VisionEncoderDecoderModel # B3 OCR
)

# Dimensionality reduction & Clustering
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import umap

# OCR evaluation 
from sklearn.metrics import accuracy_score  
from sklearn.metrics import silhouette_score

# Misc
from tqdm import tqdm

warnings.filterwarnings("ignore")
print("All imports successful")

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# ── Path for folders ──────────────────────────────────────────────────
ROOT = Path("..")
ANNOTATIONS_DIR = ROOT / "Annotations"
IMAGE_DIRS = [ROOT / f"Images{i}" for i in range(1, 5)]
TRAIN_FILE = ROOT / "training_set.txt"
VAL_FILE   = ROOT / "validation_set.txt"
TEST_FILE  = ROOT / "test_set.txt"

In [ ]:
# ── Load splits ──────────────────────────────────────────────────
def load_ids(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

train_ids = load_ids(TRAIN_FILE)
val_ids   = load_ids(VAL_FILE)
test_ids  = load_ids(TEST_FILE)

In [ ]:
# ── Build image lookup ───────────────────────────────────────────
id_to_imgs = defaultdict(list)
for img_dir in IMAGE_DIRS:
    if img_dir.exists():
        for f in img_dir.iterdir():
            if f.suffix.lower() == ".png":
                for suffix in ["_Rotation1_300dpi", "_Rotation2_300dpi"]:
                    if f.stem.endswith(suffix):
                        id_to_imgs[f.stem[:-len(suffix)]].append(f)
                        break

In [ ]:
# ── Build annotation lookup ──────────────────────────────────────
id_to_ann = defaultdict(list)
for f in ANNOTATIONS_DIR.iterdir():
    if f.suffix == ".txt":
        for suffix in ["_Rotation1_300dpi_letters", "_Rotation2_300dpi_letters"]:
            if f.stem.endswith(suffix):
                id_to_ann[f.stem[:-len(suffix)]].append(f)
                break

In [ ]:
# ── Cross-check ──────────────────────────────────────────────────
print(f"{'='*55}")
print(f"  DATASET SANITY CHECK")
print(f"{'='*55}")
print(f"  Total unique inscriptions (images):      {len(id_to_imgs)}")
print(f"  Total unique inscriptions (annotations): {len(id_to_ann)}")
print(f"{'='*55}")

all_ok = True
for split_name, ids in [("Train", train_ids), ("Val", val_ids), ("Test", test_ids)]:
    missing_img = [i for i in ids if i not in id_to_imgs]
    missing_ann = [i for i in ids if i not in id_to_ann]
    status_img = "✅" if not missing_img else "❌"
    status_ann = "✅" if not missing_ann else "❌"
    print(f"\n  {split_name} split ({len(ids)} inscriptions)")
    print(f"  {status_img} Images found:      {len(ids)-len(missing_img)}/{len(ids)}")
    print(f"  {status_ann} Annotations found: {len(ids)-len(missing_ann)}/{len(ids)}")
    if missing_img:
        print(f"     Missing images:      {missing_img[:3]}")
    if missing_ann:
        print(f"     Missing annotations: {missing_ann[:3]}")
    if missing_img or missing_ann:
        all_ok = False

print(f"\n{'='*55}")
print(f"  {'✅ ALL DATA PRESENT AND MATCHED' if all_ok else '❌ SOME DATA MISSING — CHECK ABOVE'}")
print(f"{'='*55}")

## Β1. Προεπεξεργασία Εικόνων — Image Preprocessing

Τα squeezes είναι χάρτινα αποτυπώματα λίθινων επιγραφών με ανομοιόμορφο φωτισμό,
τσαλακωμένη επιφάνεια και θόρυβο υφής — χαρακτηριστικά που δυσκολεύουν την
αυτόματη επεξεργασία. Το pipeline μας αποτελείται από 4 βήματα:

**1. Grayscale & Resize:** Οι εικόνες (~3420×1752px, 17MB) μετατρέπονται σε
grayscale (το χρώμα δεν προσφέρει πληροφορία για επιγραφές) και γίνεται resize
σε 1024×512px, μειώνοντας τη μνήμη σε ~0.5MB/εικόνα. Γίνεται πρώτο ώστε όλα
τα επόμενα βήματα να εκτελούνται στη μικρή εικόνα — αποφεύγοντας σπατάλη
υπολογιστικών πόρων.

**2. CLAHE:** Το ανομοιόμορφο φως από τις πτυχές του χαρτιού κάνει ορισμένα
τμήματα της εικόνας πιο σκοτεινά από άλλα. Το CLAHE εφαρμόζει histogram
equalization τοπικά σε 8×8 tiles, ενισχύοντας την αντίθεση ομοιόμορφα σε όλη
την εικόνα χωρίς να υπεραπλοποιεί φωτεινές περιοχές (clip_limit=2.0). Εφαρμόζεται
πριν το denoise γιατί χρειάζεται τις ακατέργαστες ακμές για να εντοπίσει
σωστά τις τοπικές διαφορές αντίθεσης.

**3. Gaussian Denoising:** Gaussian blur με kernel 5×5 αφαιρεί τον υψίσυχνο
θόρυβο από την υφή του χαρτιού, διατηρώντας τα άκρα των γραμμάτων. Εφαρμόζεται
μετά το CLAHE ώστε να μην καταστρέψει τις ακμές που χρειάζεται η ενίσχυση
αντίθεσης.

**4. Normalization:** Τα pixel values κλιμακώνονται από [0, 255] σε [0.0, 1.0]
— η τυπική μορφή εισόδου για τα deep learning μοντέλα των επόμενων σταδίων.
Γίνεται τελευταίο καθώς τα προηγούμενα βήματα λειτουργούν αποδοτικότερα με
integer values.

```
Raw Image (3420×1752, RGB, 17MB)
    → Grayscale + Resize (1024×512, 0.5MB)  # πρώτα, για αποδοτικότητα
    → CLAHE (τοπική βελτίωση αντίθεσης)     # πριν χαλάσουμε ακμές
    → Gaussian Denoise (αφαίρεση θορύβου)   # μετά την ενίσχυση
    → Normalize [0, 1]                       # τελευταίο, μετατροπή τύπου
```

In [ ]:
TARGET_H = 1024
TARGET_W = 512

def preprocess_image(image_path):
    """
    Full preprocessing pipeline:
    1. Load & convert to grayscale
    2. Resize to TARGET_H x TARGET_W
    3. CLAHE (clip=2.0, tile=8x8)
    4. Gaussian Denoising (kernel=5x5)
    5. Normalize to [0, 1]
    """
    img = cv2.imread(str(image_path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(resized)
    denoised = cv2.GaussianBlur(enhanced, (5, 5), 0)
    normalized = denoised.astype(np.float32) / 255.0
    return normalized

In [ ]:
# ── Apply to all training images ─────────────────────────────────
train_processed = {}
for insc_id in train_ids:
    paths = sorted(id_to_imgs[insc_id])
    train_processed[insc_id] = [preprocess_image(p) for p in paths]

print(f"   Preprocessed {len(train_processed)} inscriptions")
print(f"   Shape: {list(train_processed.values())[0][0].shape}")
print(f"   Dtype: {list(train_processed.values())[0][0].dtype}")
print(f"   Pixel range: [{list(train_processed.values())[0][0].min():.2f}, "
      f"{list(train_processed.values())[0][0].max():.2f}]")

In [ ]:
# ── Full pipeline visualization on one sample ─────────────────────
sample_id = train_ids[0]
sample_path = sorted(id_to_imgs[sample_id])[0]

img      = cv2.imread(str(sample_path))
gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
resized  = cv2.resize(gray, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(resized)
denoised = cv2.GaussianBlur(enhanced, (5,5), 0)
final    = denoised.astype(np.float32) / 255.0

steps = [
    ("1. Grayscale\n& Resize", resized),
    ("2. CLAHE",               enhanced),
    ("3. Denoising",           denoised),
    ("4. Normalized\n[0,1]",   final),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 7))
for col, (label, img_step) in enumerate(steps):
    axes[col].imshow(img_step, cmap="gray")
    axes[col].set_title(label, fontsize=11)
    axes[col].axis("off")

plt.suptitle(f"Full Preprocessing Pipeline — {sample_id}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Β2. Ομαδοποίηση Εικόνων μέσω Embeddings

Στόχος είναι η ομαδοποίηση των εικόνων βάσει οπτικής ομοιότητας χρησιμοποιώντας
embeddings από προ-εκπαιδευμένο οπτικό μοντέλο (DINOv2). Το pipeline αποτελείται από:

1. **Εξαγωγή embeddings** με DINOv2 από όλες τις εικόνες
2. **PCA** για αρχική μείωση διαστάσεων
3. **UMAP** για visualization στον 2D χώρο
4. **K-Means** για clustering
5. **Ονοματοδοσία** clusters μέσω LLM
6. **Ανάλυση** — εξετάζουμε αν ζεύγη εικόνων και εικόνες από τον ίδιο τόπο
   βρίσκονται στο ίδιο cluster

In [ ]:
# Load ViT model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

processor_dino = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model_dino = AutoModel.from_pretrained('facebook/dinov2-base')
model_dino.eval()
model_dino.to(device)

In [ ]:
# Extract DINOv2 Embeddings (Rotation1 + Rotation2)
def extract_embedding(img_array):
    """Extract 768-dim CLS token embedding from a preprocessed grayscale image."""
    img_uint8 = (img_array * 255).astype(np.uint8)
    img_rgb = np.stack([img_uint8] * 3, axis=-1)  # grayscale → RGB (DINOv2 expects 3 channels)
    inputs = processor_dino(images=img_rgb, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_dino(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]  # CLS token → (1, 768)
    return embedding.squeeze().cpu().numpy()  # (768,)

# Extract embeddings for both rotations
# Rotation1 → used for clustering
# Rotation2 → used for pair analysis
embeddings_rot1 = {}
embeddings_rot2 = {}

for insc_id in tqdm(train_ids, desc="Extracting embeddings"):
    imgs = train_processed[insc_id]
    embeddings_rot1[insc_id] = extract_embedding(imgs[0])  # Rotation1
    embeddings_rot2[insc_id] = extract_embedding(imgs[1])  # Rotation2

# Stack Rotation1 into matrix for PCA/UMAP/KMeans
emb_matrix = np.stack([embeddings_rot1[id_] for id_ in train_ids])
print(f"   Embeddings extracted")
print(f"   Shape: {emb_matrix.shape}")  # (181, 768)

In [ ]:
# PCA
pca = PCA(n_components=50)
emb_pca = pca.fit_transform(emb_matrix)

print(f"   PCA applied")
print(f"   Shape: {emb_pca.shape}")  # (181, 50)
print(f"   Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# UMAP
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
emb_umap = reducer.fit_transform(emb_pca)

print(f"   UMAP applied")
print(f"   Shape: {emb_umap.shape}")


# Visualize UMAP 
plt.figure(figsize=(10, 8))
plt.scatter(emb_umap[:, 0], emb_umap[:, 1], alpha=0.7, s=50)

for i, insc_id in enumerate(train_ids):
    plt.annotate(insc_id, (emb_umap[i, 0], emb_umap[i, 1]), 
                 fontsize=5, alpha=0.6)

plt.title("UMAP — DINOv2 Embeddings of Squeeze Images", fontsize=13)
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.tight_layout()
plt.show()

In [ ]:
# Try shillouette score to see the best amount of clusters
silhouette_scores = []
K = range(2, 11)

for k in K:
    kmeans_temp = KMeans(n_clusters=k, random_state=42)
    labels = kmeans_temp.fit_predict(emb_umap)  # consistent with our clustering space
    silhouette_scores.append(silhouette_score(emb_umap, labels))

plt.figure(figsize=(8, 4))
plt.plot(K, silhouette_scores, marker='o')
plt.title('Silhouette Score')
plt.xlabel('Number of clusters')
plt.ylabel('Score (higher is better)')
plt.tight_layout()
plt.show()

best_k = list(K)[silhouette_scores.index(max(silhouette_scores))]
print(f"Best k by silhouette: {best_k}")

Το Silhouette Score μεγιστοποιείται στο k=7, ωστόσο επιλέγουμε **k=3** βάσει
των οδηγιών της εργασίας. Το K-Means εφαρμόστηκε στον 2D χώρο του UMAP ώστε
τα clusters να είναι συνεπή με την οπτική απεικόνιση. Εάν το K-Means τρέξει
στον PCA χώρο (50 διαστάσεις), τα cluster boundaries δεν αντιστοιχούν στην
2D απεικόνιση λόγω της μη-γραμμικότητας του UMAP.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
cluster_labels = kmeans.fit_predict(emb_umap)

print(f"K-Means applied with k=3")
for i in range(3):
    print(f"   Cluster {i}: {np.sum(cluster_labels == i)} images")

In [ ]:
# UMAP by cluster
colors = ['#e41a1c', '#377eb8', '#4daf4a']

plt.figure(figsize=(10, 8))
for cluster_id in range(3):
    mask = cluster_labels == cluster_id
    plt.scatter(emb_umap[mask, 0], emb_umap[mask, 1],
                c=colors[cluster_id], label=f"Cluster {cluster_id}",
                alpha=0.7, s=50)

plt.title("UMAP — Colored by Cluster", fontsize=13)
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 3 images closest to each centroid

centroids = kmeans.cluster_centers_
distances = cdist(emb_umap, centroids)

fig, axes = plt.subplots(3, 3, figsize=(12, 16))

for cluster_id in range(3):
    cluster_mask = np.where(cluster_labels == cluster_id)[0]
    closest = cluster_mask[np.argsort(distances[cluster_mask, cluster_id])[:3]]
    
    for col, idx in enumerate(closest):
        insc_id = train_ids[idx]
        img = train_processed[insc_id][0]
        axes[cluster_id, col].imshow(img, cmap="gray")
        axes[cluster_id, col].set_title(f"{insc_id}", fontsize=7)
        axes[cluster_id, col].axis("off")
    
    axes[cluster_id, 0].set_ylabel(f"Cluster {cluster_id}",
                                    fontsize=11, fontweight="bold")

plt.suptitle("3 Images Closest to Each Centroid", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Ονοματοδοσία Clusters με LLM

Χρησιμοποιήσαμε το Claude (claude.ai) για την ονοματοδοσία των clusters.
Στείλαμε τις 3 εικόνες κοντά σε κάθε centroid με το prompt:

**Prompt:** *"Κοίτα αυτές τις 3 εικόνες squeeze αρχαίων ελληνικών επιγραφών
που ανήκουν στο ίδιο cluster. Δώσε ένα σύντομο περιγραφικό όνομα (max 5 λέξεις)
βάσει των οπτικών χαρακτηριστικών τους."*

**Αποτελέσματα:**
- **Cluster 0:** Dense Multi-Shape Text Inscriptions
- **Cluster 1:** Faint Oval Low-Contrast Inscriptions
- **Cluster 2:** Heavily Degraded Noisy Inscriptions

In [ ]:
# ── Helper functions for filename analysis ────────────────────────

def get_archive(insc_id):
    """Extract archive/corpus from filename (IG_II2, IAS, Agora)."""
    if insc_id.startswith("IG_II2"):
        return "IG_II2"
    elif insc_id.startswith("IAS"):
        return "IAS"
    elif insc_id.startswith("Agora"):
        return "Agora"
    else:
        return "Other"

def get_ig_number(insc_id):
    """Extract inscription number from IG_II2 filename."""
    match = re.search(r'IG_II2_(\d+)', insc_id)
    if match:
        return int(match.group(1))
    return None



In [ ]:
archive_colors = {
    'IG_II2': '#e41a1c',
    'IAS': '#377eb8',
    'Agora': '#4daf4a',
    'Other': '#984ea3'
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for cluster_id in range(3):
    cluster_mask = np.where(cluster_labels == cluster_id)[0]
    archives = [get_archive(train_ids[i]) for i in cluster_mask]
    
    archive_counts = pd.Series(archives).value_counts()
    colors = [archive_colors[a] for a in archive_counts.index]
    
    axes[cluster_id].bar(archive_counts.index, archive_counts.values, color=colors)
    axes[cluster_id].set_title(f"Cluster {cluster_id}\n({len(cluster_mask)} images)")
    axes[cluster_id].set_xlabel("Archive")
    axes[cluster_id].set_ylabel("Count")

plt.suptitle("Archive Distribution per Cluster", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


for cluster_id in range(3):
    cluster_mask = np.where(cluster_labels == cluster_id)[0]
    archives = [get_archive(train_ids[i]) for i in cluster_mask]
    print(f"Cluster {cluster_id}: {pd.Series(archives).value_counts().to_dict()}")

In [ ]:
# Archive distribution per cluster (k=3)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for cluster_id in range(3):
    cluster_mask = np.where(cluster_labels == cluster_id)[0]
    ig_numbers = [get_ig_number(train_ids[i]) for i in cluster_mask]
    ig_numbers = [n for n in ig_numbers if n is not None]
    
    axes[cluster_id].hist(ig_numbers, bins=15, color='steelblue', edgecolor='white')
    axes[cluster_id].set_title(f"Cluster {cluster_id} — IG Numbers")
    axes[cluster_id].set_xlabel("IG II² Number")
    axes[cluster_id].set_ylabel("Count")

plt.suptitle("Inscription Number Distribution per Cluster", fontsize=13)
plt.tight_layout()
plt.show()

- Η ανάλυση των archives δείχνει ότι και τα 3 clusters κυριαρχούνται από το IG_II2 corpus,
με το Cluster 2 να περιέχει αξιόλογο αριθμό IAS εικόνων (9) και το Cluster 0 λιγότερες (7),
υποδηλώνοντας ότι η οπτική ομοιότητα δεν σχετίζεται με την προέλευση αρχείου.

- Η ανάλυση των IG αριθμών δείχνει ότι τα clusters δεν αντιστοιχούν καθαρά σε εποχές:
το Cluster 0 εμφανίζει bimodal κατανομή με κενό στην Ελληνιστική περίοδο (500–1500),
το Cluster 1 κατανέμεται ομοιόμορφα με spike στη Ρωμαϊκή (>1750), ενώ το Cluster 2
κυριαρχείται από Ρωμαϊκές επιγραφές (>1750) με ελάχιστη παρουσία Ελληνιστικών.

In [ ]:
def get_inscription_base(insc_id):
    """Extract base inscription number for pair detection."""
    # Εξαγωγή του αριθμού από IG_II2 filenames
    match = re.search(r'IG_II2_(\d+)', insc_id)
    if match:
        return f"IG_II2_{match.group(1)}"
    # IAS και Agora δεν έχουν pairs
    return insc_id

# Βρες pairs
df_clusters = pd.DataFrame({
    'filename': train_ids,
    'cluster': cluster_labels,
    'base_id': [get_inscription_base(id_) for id_ in train_ids]
})

# Κράτα μόνο bases που εμφανίζονται >1 φορά
pair_bases = df_clusters.groupby('base_id').filter(lambda x: len(x) > 1)
print(f"Pairs βρέθηκαν: {pair_bases['base_id'].nunique()} inscriptions")
print()

# Για κάθε pair, έλεγξε αν είναι στο ίδιο cluster
for base, group in pair_bases.groupby('base_id'):
    clusters = group['cluster'].tolist()
    same = len(set(clusters)) == 1
    status = "✅ ίδιο cluster" if same else "❌ διαφορετικά clusters"
    print(f"{base}: {group['filename'].tolist()} → clusters {clusters} {status}")

### Ανάλυση Pairs

Εντοπίστηκαν 15 επιγραφές με πολλαπλές εικόνες (pairs). 
Από αυτές, **7/15 (47%)** βρίσκονται στο ίδιο cluster, 
ενώ **8/15 (53%)** διαχωρίζονται σε διαφορετικά clusters.

Το μοντέλο εντοπίζει οπτική ομοιότητα μεταξύ τμημάτων της ίδιας 
επιγραφής σε λιγότερο από τις μισές περιπτώσεις. Οι επιτυχημένες 
περιπτώσεις (IG_II2_1706, IG_II2_2037, IG_II2_2081, IG_II2_216, 
IG_II2_244, IG_II2_354, IG_II2_478) αφορούν τμήματα με παρόμοια 
οπτική υφή. Η αποτυχία στις υπόλοιπες οφείλεται κυρίως σε εικόνες 
που αναπαριστούν **διαφορετικά τμήματα** της ίδιας επιγραφής 
(π.χ. IG_II2_2089 με 6 τμήματα σε 2 clusters), όπου η τοπική 
οπτική υφή διαφέρει σημαντικά μεταξύ τμημάτων.

## Β3. Pipeline Αναγνώρισης Κειμένου

Στόχος είναι η κατασκευή ενός pipeline αναγνώρισης κειμένου (OCR) από εικόνες
squeeze. Αξιολογούμε χρησιμοποιώντας Character Error Rate (CER) σε λατινικούς
χαρακτήρες, όπως ορίζει ο επίσημος διαγωνισμός ICDAR 2026.

Το pipeline αποτελείται από:
1. **Προεπεξεργασία** εικόνων (από Β1)
2. **Baseline μοντέλα αναφοράς:**
   - TrOCR (`microsoft/trocr-base-printed`)
   - ViTSTR
3. **Βελτιωμένο μοντέλο** — της επιλογής μας
4. **Αξιολόγηση** με το επίσημο evaluation script του διαγωνισμού
5. **Σύγκριση** baseline vs βελτιωμένου μοντέλου

In [ ]:
sys.path.insert(0, str(ROOT))
from contest_evaluation import (readBoxFile, getRowBoxes, getRowTranscript, 
                                evaluate, run_evaluations, convert)

print("Evaluation script loaded")

In [ ]:
# ── Load TrOCR ────────────────────────────────────────────────────
trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
trocr_model.config.pad_token_id = trocr_processor.tokenizer.pad_token_id
trocr_model.config.decoder_start_token_id = trocr_processor.tokenizer.cls_token_id
trocr_model.eval()
trocr_model.to(device)

print("TrOCR loaded")

In [ ]:
# ── Line extraction ───────────────────────────────────────────────
def extract_line_images(insc_id, img_array, rotation=1):
    ann_files = id_to_ann[insc_id]
    ann_file = None
    for f in ann_files:
        if f'Rotation{rotation}' in f.name:
            ann_file = f
            break
    if ann_file is None:
        return []

    gangle, boxes, transcript, lines = readBoxFile(str(ann_file))
    if not boxes:
        return []

    rowlist, idlist = getRowBoxes(boxes, gangle=gangle, lines=lines)

    orig_h, orig_w = img_array.shape
    img_paths = sorted(id_to_imgs[insc_id])
    real_img = cv2.imread(str(img_paths[rotation-1]))
    real_h, real_w = real_img.shape[:2]

    scale_x = orig_w / real_w
    scale_y = orig_h / real_h

    lintran = ''.join(transcript)
    result = []

    for row_boxes, row_ids in zip(rowlist, idlist):
        line_text = ''.join([lintran[i] for i in row_ids])
        if not line_text.strip():
            continue

        all_x = [b.xnw for b in row_boxes] + [b.xne for b in row_boxes] + \
                [b.xse for b in row_boxes] + [b.xsw for b in row_boxes]
        all_y = [b.ynw for b in row_boxes] + [b.yne for b in row_boxes] + \
                [b.yse for b in row_boxes] + [b.ysw for b in row_boxes]

        x1 = max(0, int(min(all_x) * scale_x) - 5)
        y1 = max(0, int(min(all_y) * scale_y) - 5)
        x2 = min(orig_w, int(max(all_x) * scale_x) + 5)
        y2 = min(orig_h, int(max(all_y) * scale_y) + 5)

        line_img = img_array[y1:y2, x1:x2]
        if line_img.size > 0:
            result.append((line_img, line_text))

    return result

print("extract_line_images defined")

In [ ]:
# ── TrOCR inference ───────────────────────────────────────────────
def run_trocr_inference(line_img, processor, model, device):
    img_uint8 = (line_img * 255).astype(np.uint8)
    img_rgb = np.stack([img_uint8] * 3, axis=-1)
    pil_img = Image.fromarray(img_rgb)
    pixel_values = processor(images=pil_img, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("run_trocr_inference defined")

In [ ]:
# ── Zero-shot baseline: inference σε test set ─────────────────────
PRED_DIR = ROOT / "predictions_baseline"
PRED_DIR.mkdir(exist_ok=True)

test_processed = {}
for insc_id in test_ids:
    if insc_id in id_to_imgs:
        paths = sorted(id_to_imgs[insc_id])
        test_processed[insc_id] = [preprocess_image(p) for p in paths]

for insc_id in tqdm(test_ids, desc="Zero-shot inference"):
    if insc_id not in test_processed:
        continue
    img = test_processed[insc_id][0]
    lines_out = extract_line_images(insc_id, img, rotation=1)
    if not lines_out:
        continue
    predictions = []
    for line_img, _ in lines_out:
        pred = run_trocr_inference(line_img, trocr_processor, trocr_model, device)
        predictions.append(pred)
    pred_file = PRED_DIR / f"{insc_id}_transcript.txt"
    with open(pred_file, "w", encoding="utf-8") as f:
        f.write("\n".join(predictions))

print(f"Baseline predictions saved")

In [ ]:
# ── Baseline CER ──────────────────────────────────────────────────
total_errors = 0
total_chars = 0

for insc_id in test_ids:
    pred_file = PRED_DIR / f"{insc_id}_transcript.txt"
    if not pred_file.exists():
        continue
    with open(pred_file, "r", encoding="utf-8") as f:
        pred = f.read()
    ann_files = [f for f in id_to_ann[insc_id] if 'Rotation1' in f.name]
    if not ann_files:
        continue
    gt = getRowTranscript(str(ann_files[0]))
    cer, n_err, n_char = evaluate(pred, gt)
    total_errors += n_err
    total_chars += n_char

baseline_cer = total_errors / total_chars
print(f"Baseline CER (zero-shot TrOCR): {baseline_cer:.4f}")

In [ ]:
# ── Dataset για fine-tuning ───────────────────────────────────────
class SqueezeDataset(Dataset):
    def __init__(self, ids, processor, processed_imgs):
        self.samples = []
        self.processor = processor
        for insc_id in ids:
            if insc_id not in processed_imgs:
                continue
            img = processed_imgs[insc_id][0]
            for line_img, line_text in extract_line_images(insc_id, img):
                if line_text.strip():
                    self.samples.append((line_img, line_text))
        print(f"{len(self.samples)} line samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        line_img, text = self.samples[idx]
        img = Image.fromarray((line_img * 255).astype(np.uint8)).convert("RGB")
        pixel_values = self.processor(images=img, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            text, return_tensors="pt", padding="max_length",
            max_length=64, truncation=True
        ).input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return pixel_values, labels

# Preprocess val images
val_processed = {}
for insc_id in val_ids:
    if insc_id in id_to_imgs:
        paths = sorted(id_to_imgs[insc_id])
        val_processed[insc_id] = [preprocess_image(p) for p in paths]

train_dataset = SqueezeDataset(train_ids, trocr_processor, train_processed)
val_dataset   = SqueezeDataset(val_ids,   trocr_processor, val_processed)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False)

In [ ]:
# ── Fine-tuning ───────────────────────────────────────────────────
optimizer = AdamW(trocr_model.parameters(), lr=5e-5)
ACCUMULATION_STEPS = 2

for epoch in range(3):
    # Train
    trocr_model.train()
    train_loss = 0
    optimizer.zero_grad()
    for i, (pixel_values, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/3")):
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        loss = trocr_model(pixel_values=pixel_values, labels=labels).loss / ACCUMULATION_STEPS
        loss.backward()
        if (i + 1) % ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
        train_loss += loss.item() * ACCUMULATION_STEPS

    # Validate
    trocr_model.eval()
    val_loss = 0
    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            val_loss += trocr_model(pixel_values=pixel_values, labels=labels).loss.item()

    print(f"Epoch {epoch+1} — Train: {train_loss/len(train_loader):.4f} | Val: {val_loss/len(val_loader):.4f}")

print("✅ Fine-tuning complete")

In [ ]:
# ── Fine-tuned inference σε test set ─────────────────────────────
PRED_FT_DIR = ROOT / "predictions_finetuned"
PRED_FT_DIR.mkdir(exist_ok=True)

trocr_model.eval()
for insc_id in tqdm(test_ids, desc="Fine-tuned inference"):
    if insc_id not in test_processed:
        continue
    img = test_processed[insc_id][0]
    lines_out = extract_line_images(insc_id, img, rotation=1)
    if not lines_out:
        continue
    predictions = []
    for line_img, _ in lines_out:
        pred = run_trocr_inference(line_img, trocr_processor, trocr_model, device)
        predictions.append(pred)
    pred_file = PRED_FT_DIR / f"{insc_id}_transcript.txt"
    with open(pred_file, "w", encoding="utf-8") as f:
        f.write("\n".join(predictions))

print(f"Fine-tuned predictions saved")

In [ ]:
# ── Fine-tuned CER + Σύγκριση ─────────────────────────────────────
total_errors = 0
total_chars = 0

for insc_id in test_ids:
    pred_file = PRED_FT_DIR / f"{insc_id}_transcript.txt"
    if not pred_file.exists():
        continue
    with open(pred_file, "r", encoding="utf-8") as f:
        pred = f.read()
    ann_files = [f for f in id_to_ann[insc_id] if 'Rotation1' in f.name]
    if not ann_files:
        continue
    gt = getRowTranscript(str(ann_files[0]))
    cer, n_err, n_char = evaluate(pred, gt)
    total_errors += n_err
    total_chars += n_char

finetuned_cer = total_errors / total_chars

print(f"Baseline CER  (zero-shot):  {baseline_cer:.4f}")
print(f"Fine-tuned CER:             {finetuned_cer:.4f}")
print(f"Βελτίωση:                   {(baseline_cer - finetuned_cer):.4f} ({(baseline_cer - finetuned_cer)/baseline_cer*100:.1f}%)")